In [ ]:
import os
os.environ["OPENAI_API_KEY"]="YOUR_OPENAI_API_KEY"  # Replace with your actual OpenAI API key

In [3]:
!pip install -q youtube-transcript-api langchain-community langchain-openai \
               faiss-cpu tiktoken python-dotenv

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

C:\Users\msamp\AppData\Local\Temp\ipykernel_20536\4040074790.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [7]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled,RequestBlocked

Step 1a - Indexing (Document Ingestion)

In [22]:
#you have to select a video which has english transcripts otherwise this piece of code returns error
video_id = "znF2U_3Z210" # only the ID, not full URL
try:
    # If you don’t care which language, this returns the “best” one
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["en"])
    #With the current youtube-transcript-api, fetch() returns a FetchedTranscript object, which is iterable over transcript snippet/chunk objects
    #with the attributes "text", "start" and "duration".
    print(transcript_list)
    print()
    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)  #join() internally iterates through the generator.
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="What is machine learning? Well, I think you have a basic idea. Right? It's the tech that", start=0.6, duration=6.72), FetchedTranscriptSnippet(text='recommends which video to watch next on YouTube. The machine is learning your preferences', start=7.32, duration=6.76), FetchedTranscriptSnippet(text='and its bot powers chatbots. But is ML or machine', start=14.08, duration=6.28), FetchedTranscriptSnippet(text='learning synonymous with AI? Or what about deep', start=20.36, duration=6.68), FetchedTranscriptSnippet(text='learning? DL. Are deep learning and machine learning the same thing? Well, no,', start=27.04, duration=6.76), FetchedTranscriptSnippet(text="it's a hierarchy. So, machine learning is a subset of", start=33.8, duration=6.72), FetchedTranscriptSnippet(text='AI that focuses on algorithms that can learn the patterns of training data and then make', start=40.84, duration=6.919), FetchedTranscriptSnippet(text='accurate in

untill now we just loaded the full transcript i.e the document is ready

Step 1b - Indexing (Text Splitting)

In [39]:
splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

chunks = splitter.create_documents([transcript])  # create_documents() expects a list of texts
# create_documents() splits the text into chunks and converts each chunk into a Document object.

# Why convert the chunks into Document objects instead of keeping them as strings?
# We don't strictly need Document objects because strings/text can also be converted into
# embeddings, stored in a vector store, and used to build a RAG application.
# However, Document objects are useful because they store the chunk's text in `page_content`
# along with metadata, which becomes useful later in the RAG pipeline.

#simply, Embeddings work on text. LangChain Document is a convenient structure that packages the text together with metadata, which is particularly useful in RAG pipelines.

In [41]:
len(chunks)

102

In [51]:
chunks[50]

Document(metadata={}, page_content='A well-known clustering method is called k-means clustering, so you choose k-groups. Repeatedly')

In [53]:
chunks[51]

Document(metadata={}, page_content='Repeatedly assign each item to the nearest group average. Then you recompute the averages until')

Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [57]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = FAISS.from_documents(chunks, embeddings)
# Here, `chunks` should be a list of Document objects because we are using `from_documents()`.
# In other words: `documents = chunks` and `embedding_model = embeddings`.
#
# If we had used `split_text()` above and therefore had chunks as plain strings
# instead of Document objects, we would use `FAISS.from_texts(chunks, embeddings)`.

In [60]:
#index_to_docstore_id maps the position/index of a vector in the FAISS index to the ID of the corresponding Document in the docstore.
vector_store.index_to_docstore_id

{0: '5a0c57b8-7d17-4687-b1cd-f6e530d42ecf',
 1: '9f810ecb-5db9-420b-89b0-20f1bb43fc06',
 2: 'ee1accc5-052c-4794-9ebc-37f9d4a1bba7',
 3: '81a1013a-083f-4118-aa16-b40bedde74b1',
 4: 'af45fda0-b90c-4df8-ba6f-9a0c01ace600',
 5: 'c27f0a42-1648-4efa-be1f-78e1215d65ee',
 6: 'af1a6d5c-80e1-48cb-a386-d0c7fbf40705',
 7: '152df021-7816-4704-a465-8e1eb38d42df',
 8: '0e03376f-404a-4ab0-8756-9483cd552df0',
 9: '9fed1472-1963-4c90-9787-7d3305cc0501',
 10: 'b9ad4957-dda8-47b1-8371-975ca662702a',
 11: 'b237ca21-8b9d-4323-88ed-7dfb2127e540',
 12: 'af8384b2-ca1a-481c-93ab-dec1b2f64a95',
 13: '402103fd-46fa-4c7a-b35b-f47d4f70c6f4',
 14: '765b3bb3-d0ab-48c7-a25e-135119a8b913',
 15: '4ef5c475-010b-44de-a33f-58575eae6706',
 16: 'a88b06d2-b2ec-4e35-8729-4b2bc96dc8b0',
 17: 'c38a87d7-13c4-41b4-9faa-c4228f2603cc',
 18: 'bb3a90a0-6a94-45d1-83f1-77f012d5b649',
 19: 'dbaf5ad4-dded-4f61-ae65-1bea93b7f2d4',
 20: 'c58e9a4c-abc1-45a9-a959-fc5a58c37382',
 21: 'a3d9e3f8-1a47-4b60-9ecc-3a44cf3ecba4',
 22: 'b7cb19db-d8f3-

In [64]:
#if u want to see the chunk :
vector_store.get_by_ids(['6a654e6a-c7e3-4e98-888b-00710fb30868'])

[Document(id='6a654e6a-c7e3-4e98-888b-00710fb30868', metadata={}, page_content='neural network design, but they still rely on the same ML principles of pattern recognition and')]

Step 2 - Retrieval

In [71]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 10})
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000025FB47DDE20>, search_kwargs={'k': 10})

In [73]:
retriever.invoke('What is reinforcement learning?')

[Document(id='6deba74c-1022-4320-b021-b5cd41f83cbc', metadata={}, page_content="learning has made a bit of a comeback with RLHF. That's reinforcement learning with human feedback."),
 Document(id='bf25d6e9-7347-4d9d-8bfc-f89c5e0ce3b5', metadata={}, page_content='then there is also reinforcement learning, which optimizes a policy through trial and error with'),
 Document(id='bf9737b1-a12b-4e82-bd81-94ea9d5b816f', metadata={}, page_content='recognition and data and model training and inference. And in fact, even reinforcement learning has'),
 Document(id='c0d880d0-831e-4c0b-9e01-d59047d86e37', metadata={}, page_content="encoders as another example. Then there's reinforcement learning. So an agent interacts with an"),
 Document(id='152df021-7816-4704-a465-8e1eb38d42df', metadata={}, page_content='is learning pattern recognition that lets models make decisions or predictions without explicit'),
 Document(id='2f50a431-d7ee-43b9-a8dd-96c1bc61818e', metadata={}, page_content='and error with r

Step 3 - Augmentation

In [76]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [93]:
question = "is the topic reinforcement learning discussed in the video? if yes then what did we discuss?"
#try simple question -> what is supervised learning?
retrieved_docs = retriever.invoke(question)
retrieved_docs

[Document(id='1dd4c44d-d014-4593-86d4-8174a75ebfea', metadata={}, page_content="we've covered here regression, classification, clustering and reinforcement learning that stuff all"),
 Document(id='6deba74c-1022-4320-b021-b5cd41f83cbc', metadata={}, page_content="learning has made a bit of a comeback with RLHF. That's reinforcement learning with human feedback."),
 Document(id='bf9737b1-a12b-4e82-bd81-94ea9d5b816f', metadata={}, page_content='recognition and data and model training and inference. And in fact, even reinforcement learning has'),
 Document(id='bf25d6e9-7347-4d9d-8bfc-f89c5e0ce3b5', metadata={}, page_content='then there is also reinforcement learning, which optimizes a policy through trial and error with'),
 Document(id='b45926a2-bbda-42c2-b305-b7f42c98d13d', metadata={}, page_content="the rewards and the penalties help the model learn. Now, most of what we've covered here"),
 Document(id='c0d880d0-831e-4c0b-9e01-d59047d86e37', metadata={}, page_content="encoders as another

In [95]:
contextual_info = " ".join(d.page_content for d in retrieved_docs)
contextual_info

"we've covered here regression, classification, clustering and reinforcement learning that stuff all learning has made a bit of a comeback with RLHF. That's reinforcement learning with human feedback. recognition and data and model training and inference. And in fact, even reinforcement learning has then there is also reinforcement learning, which optimizes a policy through trial and error with the rewards and the penalties help the model learn. Now, most of what we've covered here encoders as another example. Then there's reinforcement learning. So an agent interacts with an with AI? Or what about deep learning? DL. Are deep learning and machine learning the same thing? as well. That's where we have multiple tags all at once. And modern supervised learning. It and error with rewards and penalties. Now there are several types of supervised machine learning the same thing? Well, no, it's a hierarchy. So, machine learning is a subset of AI that focuses on"

In [97]:
final_prompt = prompt.invoke({'question':question, 'context':contextual_info})
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      we've covered here regression, classification, clustering and reinforcement learning that stuff all learning has made a bit of a comeback with RLHF. That's reinforcement learning with human feedback. recognition and data and model training and inference. And in fact, even reinforcement learning has then there is also reinforcement learning, which optimizes a policy through trial and error with the rewards and the penalties help the model learn. Now, most of what we've covered here encoders as another example. Then there's reinforcement learning. So an agent interacts with an with AI? Or what about deep learning? DL. Are deep learning and machine learning the same thing? as well. That's where we have multiple tags all at once. And modern supervised learning. It and error with rewards and penalties. 

Step 4 - Generation:

In [100]:
llm = ChatOpenAI(model="gpt-6-luna")
result = llm.invoke(final_prompt)
result.content

'Yes. The video discusses reinforcement learning as learning through trial and error: an agent interacts with an environment, and rewards and penalties help it learn and optimize a policy. It also mentions RLHF, or reinforcement learning with human feedback.'

In [102]:
#EACH COMPONENT WAS CALLED MANUALLY (RETR., LLM AND PROMPT)

BUILDING A CHAIN

In [107]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

In [119]:
def format_docs(r_docs):
    context_text = " ".join(d.page_content for d in r_docs)
    return context_text

In [125]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs), #remember u just have to pass the function name to convert it into a runnable but don't call the function.
    #retriever gives list of docs as output u can refer to code parts high above. but let's not bother with what type of output of retriever is given
    # to format_docs as input ..point is simply retriever | format_docs can't be written since to include in a chain we have to convert it into a runnable.
    #now list of docs pass through parameter of format_docs and then u know the rest
    'question': RunnablePassthrough()
})

In [127]:
parallel_chain.invoke("what is ml discussed in entire video?")

{'context': "inside ML. Now the central premise of machine learning is that if you optimize the machine's learn hierarchical representations. So, ML sits inside AI and then DL sits inside ML. Now the preferences and its bot powers chatbots. But is ML or machine learning synonymous with AI? Or what with AI? Or what about deep learning? DL. Are deep learning and machine learning the same thing? and agentic AI, the foundations are still classic ML concepts. just scaled up and combined in new of that is LLMs or large language models. And those are built on top of something called a behavior. So, while the buzz today is all around LLMs and generative AI and agentic AI, the that stuff all kind of falls under what you might call classic machine learning. These techniques these old ideas applied in new ways. And of course, the most famous example of that is LLMs or we've covered here regression, classification, clustering and reinforcement learning that stuff all",
 'question': 'what is ml dis

In [129]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [131]:
main_chain = parallel_chain | prompt | llm | parser

In [137]:
main_chain.invoke( "can u summarize the video?")

'The video discusses several machine-learning approaches: recommendation systems that learn your preferences, regression, classification, clustering, and reinforcement learning. It gives a driving example where an agent chooses actions such as steering, braking, and accelerating, and learns from rewards for safe behavior and penalties for mistakes. It also mentions PCA and encoders for reducing the number of dimensions in data, and clustering customers into groups such as bargain hunters and loyal repeaters.'